# Day 1 · How LLMs Work & Your First API Call

This notebook implements **Today's Build** (Sweet Crumbs Home Bakery assistant) and all five experiments in **Your 10%** from the Day 1 AI Agents Workshop tutorial.

In [ ]:
# Step 0: Install the OpenAI SDK (Groq uses an OpenAI-compatible API format)
!pip install -q openai

In [ ]:
import os
from openai import OpenAI

# Initialize the Groq client using the provided API key
# (Tip: In production or public repos, store keys in Colab Secrets or environment variables)
GROQ_API_KEY = "your_groq_api_key_here"  # Replace with your

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

MODEL = "openai/gpt-oss-20b"
print("Client configured successfully for Groq!")

---
## Today's Build: Sweet Crumbs' First Answer
Using a structured `SYSTEM` prompt (`ROLE`, `FACTS`, `RULES`, `FORMAT`) and a low temperature (`0.2`) so answers stay accurate and grounded in bakery rules.

In [ ]:
SYSTEM = """ROLE:   You are the assistant for Sweet Crumbs Home Bakery.
FACTS:  We bake to order. Orders need 24 hours' notice. All cakes have eggless versions.
        Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80.
RULES:  Never promise a delivery time. If the answer isn't in FACTS, say "I'll check with the owner."
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

response = client.chat.completions.create(
    model=MODEL,
    temperature=0.2,  # facts and prices: keep it focused
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "Can I get an eggless cake today evening?"},
    ],
)

print("Reply:", response.choices[0].message.content)
print("Usage:", response.usage)

---
## Task 1: Temperature
Ask *"Write a tagline for Sweet Crumbs"* three times at `temperature=0.2` (focused/repeatable) and three times at `temperature=1.2` (creative/varied).

In [ ]:
prompt_tagline = "Write a one-line tagline for Sweet Crumbs Home Bakery."

print("=== Low Temperature (0.2) ===")
for i in range(3):
    res = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        messages=[
            {"role": "system", "content": "You are a helpful marketing copywriter."},
            {"role": "user", "content": prompt_tagline},
        ],
    )
    print(f"Run {i+1}: {res.choices[0].message.content.strip()}")

print("\n=== High Temperature (1.2) ===")
for i in range(3):
    res = client.chat.completions.create(
        model=MODEL,
        temperature=1.2,
        messages=[
            {"role": "system", "content": "You are a helpful marketing copywriter."},
            {"role": "user", "content": prompt_tagline},
        ],
    )
    print(f"Run {i+1}: {res.choices[0].message.content.strip()}")

---
## Task 2: Hallucination
Delete the `FACTS` line and ask *"Do you deliver to Kakinada?"* to see what the model invents when not grounded.

In [ ]:
SYSTEM_NO_FACTS = """ROLE:   You are the assistant for Sweet Crumbs Home Bakery.
FORMAT: Under 50 words, friendly."""

res_hallucination = client.chat.completions.create(
    model=MODEL,
    temperature=0.7,
    messages=[
        {"role": "system", "content": SYSTEM_NO_FACTS},
        {"role": "user", "content": "Do you deliver to Kakinada? How much is the delivery fee?"},
    ],
)

print("Without FACTS (may hallucinate):")
print(res_hallucination.choices[0].message.content)

---
## Task 3: No Memory
1. Make two separate stateless calls (`"My name is Ravi"`, then `"What is my name?"`).
2. Put the conversation history into one `messages` list and try again.

In [ ]:
# Part A: Two separate calls (Stateless - model forgets)
call_1 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "My name is Ravi."}],
)
print("Call 1 reply:", call_1.choices[0].message.content)

call_2 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is my name?"}],
)
print("Call 2 reply (no history):", call_2.choices[0].message.content)

# Part B: Resending full conversation history
history = [
    {"role": "user", "content": "My name is Ravi."},
    {"role": "assistant", "content": call_1.choices[0].message.content},
    {"role": "user", "content": "What is my name?"},
]
call_with_memory = client.chat.completions.create(
    model=MODEL,
    messages=history,
)
print("\nCall with full history:", call_with_memory.choices[0].message.content)

---
## Task 4: Tokens (English vs. Telugu)
Ask the same question in English and in Telugu and compare `response.usage.prompt_tokens`.

In [ ]:
q_en = "I want to order 2 eggless chocolate cakes for tomorrow evening."
q_te = "రేపు సాయంత్రం కోసం నాకు 2 ఎగ్‌లెస్ చాక్లెట్ కేకులు ఆర్డర్ చేయాలి."

res_en = client.chat.completions.create(
    model=MODEL,
    temperature=0.2,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": q_en},
    ],
)

res_te = client.chat.completions.create(
    model=MODEL,
    temperature=0.2,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": q_te},
    ],
)

print(f"English prompt tokens: {res_en.usage.prompt_tokens} | Completion tokens: {res_en.usage.completion_tokens}")
print("English reply:", res_en.choices[0].message.content)
print("-" * 60)
print(f"Telugu prompt tokens:  {res_te.usage.prompt_tokens} | Completion tokens: {res_te.usage.completion_tokens}")
print("Telugu reply:", res_te.choices[0].message.content)

---
## Task 5: Your Business
Rewrite `SYSTEM` for a custom business (5 facts, 3 rules) and ask 3 real customer questions. Below is an example for a **Campus Tech & Laptop Repair Desk**.

In [ ]:
MY_BUSINESS_SYSTEM = """ROLE:   You are the front-desk assistant for ByteFix Campus Laptop & Tech Care.
FACTS:  1. Open Monday to Saturday, 9:30 AM to 6:30 PM (closed Sundays).
        2. Basic OS reinstallation and driver setup costs Rs 300.
        3. Thermal paste repasting and internal dust cleaning costs Rs 450.
        4. Screen and battery replacements require 48 hours for part procurement.
        5. Free 15-minute hardware diagnostics for college students with valid ID.
RULES:  1. Never guarantee same-day hardware part replacement.
        2. Never guess repair prices for liquid damage; ask them to bring the laptop in.
        3. If a question isn't covered in FACTS, say "I'll check with our senior technician."
FORMAT: Concise, helpful, under 50 words."""

customer_questions = [
    "How much do you charge for laptop dust cleaning and thermal repasting?",
    "I spilled water on my keyboard just now, what will the repair cost?",
    "Can you replace my laptop battery within 2 hours today?",
]

for idx, q in enumerate(customer_questions, 1):
    res = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        messages=[
            {"role": "system", "content": MY_BUSINESS_SYSTEM},
            {"role": "user", "content": q},
        ],
    )
    print(f"Q{idx}: {q}")
    print(f"A{idx}: {res.choices[0].message.content}\n")